In [1]:
from openai import OpenAI

In [3]:
import os
NIM_API_KEY = os.getenv("NIM_API_KEY")
client= OpenAI(
    api_key=NIM_API_KEY,
    base_url="https://integrate.api.nvidia.com/v1"
)

In [5]:
def get_streaming_response(input_query):
    completion=client.chat.completions.create(
        model="deepseek-ai/deepseek-r1",
        messages=[
            {
                "role": "user",
                "content": input_query
            }
        ],
        temperature=0.6,
        stream=True,
        max_tokens=4096,
        top_p=0.7
    )
    
    for chunk in completion:
        if chunk.choices[0].delta.content is not None:
            print(chunk.choices[0].delta.content,end="")


In [ ]:
question_1="How many solutions does the equation x^2 + 4x + 4 = 0 have?"
get_streaming_response(question_1)

<think>
Okay, so I need to figure out how many solutions the equation x² + 4x + 4 = 0 has. Hmm, let's start by recalling what I know about quadratic equations. A quadratic equation is of the form ax² + bx + c = 0, and the number of solutions it has depends on the discriminant, which is b² - 4ac. If the discriminant is positive, there are two real solutions; if it's zero, there's exactly one real solution; and if it's negative, there are two complex solutions. 

Alright, so let's identify the coefficients here. Comparing x² + 4x + 4 = 0 with the standard form, a is 1, b is 4, and c is 4. Let me write that down:

a = 1
b = 4
c = 4

Now, calculating the discriminant: b² - 4ac. Plugging in the values:

Discriminant = (4)² - 4*(1)*(4) = 16 - 16 = 0

Oh, the discriminant is zero. That means there's exactly one real solution, right? But wait, sometimes people say that there are two real solutions but they are the same, so it's a repeated root. Hmm, so technically, does that count as one solut

In [4]:
"""
solve_math_problem:
input: str - The math problem to solve
output: dict - contains
    - solution: str - Step-by-step solution
    - answer: str - Final numerical answer
    - calculated: bool - True if successfully calculated, False otherwise
"""

from openai import OpenAI
import os
import re
import json
from typing import Dict, Any

class MathProblemSolver:
    """
    A module for solving math problems using DeepSeek R1 model via NVIDIA NIM API.
    
    This class provides a clean interface for solving mathematical problems
    and returning structured results with solution steps and final answers.
    """
    
    def __init__(self, api_key: str = None):
        """
        Initialize the MathProblemSolver.
        
        Args:
            api_key (str, optional): NIM API key. If not provided, will try to get from environment.
        """
        self.api_key = api_key or os.getenv("NIM_API_KEY")
        if not self.api_key:
            raise ValueError("NIM_API_KEY must be provided either as parameter or environment variable")
        
        self.client = OpenAI(
            api_key=self.api_key,
            base_url="https://integrate.api.nvidia.com/v1"
        )
    
    def solve_math_problem(self, query: str) -> Dict[str, Any]:
        """
        Solve a math problem and return structured results.
        
        Args:
            query (str): The math problem to solve
            
        Returns:
            Dict[str, Any]: Dictionary containing:
                - solution (str): Step-by-step solution
                - answer (str): Final numerical answer
                - calculated (bool): True if successfully calculated, False otherwise
        """
        try:
            # Enhanced prompt for better structured output
            enhanced_prompt = f"""
            Solve the following math problem step by step. Please provide:
            1. A clear step-by-step solution
            2. The final answer or expression as the answer (give a numerical value if there are any constants like pi, e, etc.)
            
            Problem: {query}
            
            Please format your response clearly with steps and highlight the final answer or expression.
            """
            
            # Get streaming response
            full_response = self._get_streaming_response(enhanced_prompt)
            
            if not full_response:
                return {
                    "solution": "No response received from the model",
                    "answer": "N/A",
                    "calculated": False
                }
            
            # Parse the response to extract solution and answer
            solution, answer, calculated = self._parse_response(full_response)
            
            return {
                "solution": solution,
                "answer": answer,
                "calculated": calculated
            }
            
        except Exception as e:
            return {
                "solution": f"Error occurred: {str(e)}",
                "answer": "N/A",
                "calculated": False
            }
    
    def _get_streaming_response(self, input_query: str) -> str:
        """
        Get streaming response from the model and return complete response.
        
        Args:
            input_query (str): The query to send to the model
            
        Returns:
            str: Complete response from the model
        """
        try:
            completion = self.client.chat.completions.create(
                model="deepseek-ai/deepseek-r1",
                messages=[
                    {
                        "role": "user",
                        "content": input_query
                    }
                ],
                temperature=0.6,
                stream=True,
                max_tokens=4096,
                top_p=0.7
            )
            
            full_response = ""
            for chunk in completion:
                if chunk.choices[0].delta.content is not None:
                    full_response += chunk.choices[0].delta.content
            
            return full_response
            
        except Exception as e:
            print(f"Error in streaming response: {e}")
            return ""
    
    def _parse_response(self, response: str) -> tuple:
        """
        Parse the model response to extract solution steps and final answer.
        
        Args:
            response (str): Raw response from the model
            
        Returns:
            tuple: (solution, answer, calculated)
        """
        try:
            # Clean up the response
            response = response.strip()
            
            # Try to find the final answer using various patterns
            answer_patterns = [
                r'final answer[:\s]*([+-]?\d*\.?\d+)',
                r'answer[:\s]*([+-]?\d*\.?\d+)',
                r'result[:\s]*([+-]?\d*\.?\d+)',
                r'solution[:\s]*([+-]?\d*\.?\d+)',
                r'=\s*([+-]?\d*\.?\d+)\s*$',
                r'([+-]?\d*\.?\d+)\s*$'
            ]
            
            answer = "N/A"
            calculated = False
            
            for pattern in answer_patterns:
                matches = re.findall(pattern, response, re.IGNORECASE | re.MULTILINE)
                if matches:
                    # Get the last match (usually the final answer)
                    answer = matches[-1]
                    calculated = True
                    break
            
            # If no numerical answer found, try to extract from the end of response
            if not calculated:
                # Look for numbers at the end of the response
                numbers = re.findall(r'([+-]?\d*\.?\d+)', response)
                if numbers:
                    answer = numbers[-1]
                    calculated = True
            
            # The solution is the entire response
            solution = response
            
            return solution, answer, calculated
            
        except Exception as e:
            return f"Error parsing response: {str(e)}", "N/A", False
    
    def solve_batch(self, queries: list) -> list:
        """
        Solve multiple math problems in batch.
        
        Args:
            queries (list): List of math problems to solve
            
        Returns:
            list: List of result dictionaries
        """
        results = []
        for query in queries:
            result = self.solve_math_problem(query)
            results.append(result)
        return results


# Convenience function for external use
def solve_math_problem(query: str, api_key: str = None) -> Dict[str, Any]:
    """
    Convenience function to solve a single math problem.
    
    Args:
        query (str): The math problem to solve
        api_key (str, optional): NIM API key
        
    Returns:
        Dict[str, Any]: Result dictionary with solution, answer, and calculated status
    """
    solver = MathProblemSolver(api_key)
    return solver.solve_math_problem(query)


# Example usage and testing
if __name__ == "__main__":
    # Example usage
    solver = MathProblemSolver()
    
    # Test problems
    test_problems = [
        "Find the area of a circle with radius 7",
        "Calculate the derivative of x^2 + 3x + 2"
    ]
    
    print("Testing Math Problem Solver:")
    print("=" * 50)
    
    for i, problem in enumerate(test_problems, 1):
        print(f"\nProblem {i}: {problem}")
        result = solver.solve_math_problem(problem)
        
        print(f"Calculated: {result['calculated']}")
        print(f"Answer: {result['answer']}")
        print(f"Solution: {result['solution']}")
        print("-" * 30)
    

Testing Math Problem Solver:

Problem 1: Find the area of a circle with radius 7
Calculated: True
Answer: 49
Solution: <think>
Okay, so I need to find the area of a circle with radius 7. Hmm, let me think. I remember that the formula for the area of a circle is something with pi and radius squared. Let me recall... Oh right, the formula is A = πr². Yeah, that's it. So the radius here is given as 7. 

Wait, let me make sure I got the formula right. Area equals pi times radius squared. Yes, that's correct. So substituting the radius into the formula should give me the area. Let me write that down step by step.

First, write down the formula: A = πr². Then plug in the radius value. The radius r is 7, so replacing r with 7 in the formula gives A = π*(7)². Now, I need to calculate 7 squared. 7 times 7 is 49. So the area becomes A = π*49. 

So the area is 49π. But wait, sometimes problems might ask for a numerical value. The question here just says to find the area, and it mentions to includ